# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: Kamlesh Appasaheb Kale
#### Student ID:S4033538

Environment: Python 3 and Jupyter notebook

Libraries used: Below are the main libraries I used in my assignment:

* pandas
* numpy
* sklearn.feature_extraction.text.CountVectorizer
* gensim.downloader
* sklearn.feature_extraction.text.TfidfVectorizer
* sklearn.model_selection.KFold
* sklearn.linear_model.LogisticRegression
* nltk.tokenize.RegexpTokenizer
* nltk.corpus.stopwords
* nltk.probability.FreqDist

## Introduction
In this assignment, I worked on comparing different text features from customer reviews to see how adding more information affects the accuracy of a classification model. The dataset I used had multiple features like the Clothing ID, Age, Review Title, and Detailed Review Text. My main goal was to find out if including just the title, just the review text, or both together would give me better accuracy when predicting whether customers would recommend a product.

To do this, I tried three different setups: one using only the title, one using only the review text, and one combining both. I cleaned up each text feature by removing unnecessary words, lowercasing everything, and using a Count Vectorizer to convert the text into numerical features. Then, I trained a Logistic Regression model on each setup and used 5-fold cross-validation to evaluate my results. This way, I could see how each feature representation performed and whether combining the title and review text improved my model’s prediction accuracy.

## Importing libraries 

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import gensim.downloader as api
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.probability import FreqDist

C:\Users\Kamlesh\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Kamlesh\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


# Task 2. Generating Feature Representations for Clothing Items Reviews

### Load Processed Data
Here, we’re starting by loading our cleaned review data from a CSV file called `processed.csv.` This file has our reviews already cleaned up and ready for us to work with. Then, we make sure there are no missing values and convert the reviews into a list format. This gives us a clear starting point to transform the reviews into something our models can understand.

In [2]:
data_df = pd.read_csv('processed.csv') # loading the processed CSV file containing cleaned reviews

reviews = data_df['Final Cleaned Review'].dropna().tolist()  # removing NaN values and convert to list

print(f"Number of reviews loaded: {len(reviews)}") # verifying the first few reviews to make sure the data is loaded correctly
print("Sample Reviews:", reviews[:3])  # showing the first 3 reviews as a sample

Number of reviews loaded: 19652
Sample Reviews: ['high hopes wanted work initially petite usual found outrageously fact zip reordered petite medium half nicely bottom half tight layer cheap net layers imo major design flaw net layer sewn directly zipper', 'jumpsuit fun flirty fabulous time compliments', 'shirt due adjustable front tie length leggings sleeveless pairs cardigan shirt']


### Load the Vocabulary
This line reads our vocabulary from the file `vocab.txt`, which lists the words we want to consider and their unique identifiers. Each line in the file looks like `word:index` (e.g., `shirt:0`). We split each line at the colon (`:`) to create a dictionary `vocab` where each word is mapped to its index. This dictionary tells `CountVectorizer` and `TfidfVectorizer` which words to look for, ensuring our representation is specific to our selected words.

In [3]:
with open('vocab.txt', 'r') as vocab_file: # loading the vocabulary from vocab.txt
    vocab = {line.split(':')[0]: int(line.split(':')[1]) for line in vocab_file}

# 2.1 Bag-of-words model

### 2.1.1 Initialize CountVectorizer and Generate Count Vectors
Now, we’re using the `CountVectorizer` to transform our reviews into count vectors. What’s key here is that `CountVectorizer` is using our custom vocabulary to create these vectors, so every review is represented by how many times each word appears. When we print the shape `(19662, 7529)`, it tells us that we have 19,662 reviews and 7,529 unique words in our vocabulary. This shows us that every review is now a vector of length 7,529, where each number represents the count of a word.

In [4]:
cVectorizer = CountVectorizer(analyzer="word", vocabulary=vocab) # initializing CountVectorizer with the predefined vocabulary

count_features = cVectorizer.fit_transform(reviews) # transform the reviews into count vector features
print(count_features.shape)

(19652, 7529)


## 2.2 Models based on word embeddings

### 2.2.1 Load a pre-trained Word2Vec model

We begin by loading the pre-trained Word2Vec model from the Google News dataset. This model is quite powerful and contains around 3 million word vectors, each having 300 dimensions. This means that the model maps a large number of words and phrases to numerical vectors. We use `api.load('word2vec-google-news-300')` to load the model, and then confirm by printing `"Loaded the pre-trained Word2Vec model."`. Wwe can easily check the vector size by referring to its `vector_size` attribute:

This model can understand word similarities and relationships, which is helpful for creating meaningful word embeddings. However, since it’s a large model, it requires considerable memory. That's why loading it might take some time, depending on our system resources.

In [5]:
word2vec_model = api.load('word2vec-google-news-300') # here loading the pre-trained Word2Vec model (Google News 300)
print("Loaded the pre-trained Word2Vec model.")

Loaded the pre-trained Word2Vec model.


In [6]:
word2vec_model.vector_size # we can easily check the vector size by referring to its vector_size attribute:

300

### 2.2.2 Generate Unweighted Word Embeddings

Next, we generate unweighted word embeddings for our reviews using the `get_unweighted_embeddings` function. This function goes through each review in our dataset, splits the review into words, and then checks if each word is present in our Word2Vec model.

For words that are in the model, we extract the corresponding word vector. We then calculate the average of all word vectors in a review to form a single vector representation for that review using `np.mean(word_vectors, axis=0)`. If a review doesn’t contain any words from our model, we set its vector to zeros using `np.zeros(model.vector_size)`. Finally, the result is stored in an array format using `np.array(review_vectors)`.

By converting the result to an array, we can easily analyze and perform operations on the embeddings. After this step, we print the shape of our unweighted embeddings to confirm its dimensions: `(19652, 300)`. This indicates that we have 19,652 reviews, and each is represented using a 300-dimensional vector.

In [7]:
def get_unweighted_embeddings(reviews, model): # generating Unweighted Embeddings
    review_vectors = []
    for review in reviews:
        word_vectors = [model[word] for word in review.split() if word in model]
        if word_vectors:
           
            review_vector = np.mean(word_vectors, axis=0)  # average of word vectors
        else:
            review_vector = np.zeros(model.vector_size)  # if no words are in the model
        review_vectors.append(review_vector)
    return np.array(review_vectors)

unweighted_embeddings = get_unweighted_embeddings(reviews, word2vec_model) # generate unweighted embeddings 
print("Unweighted Embeddings Shape:", unweighted_embeddings.shape)

Unweighted Embeddings Shape: (19652, 300)


In [8]:
print("Sample Unweighted Embedding for Review 1:\n", unweighted_embeddings[0][:10])  # here we print first 10 sample of unweighted embeddings values

Sample Unweighted Embedding for Review 1:
 [-0.00439262  0.02944088 -0.04074097  0.08839417 -0.04631996 -0.04612923
  0.0890007  -0.08046263  0.06069422  0.09231472]


### 2.2.3 Generate TF-IDF Weighted Word Embeddings

#### Calculating TF-IDF Scores for Reviews

Before we proceed with generating TF-IDF weighted embeddings, we need to compute the TF-IDF scores for each word in our reviews. We use `TfidfVectorizer` and pass our custom vocabulary `(vocab)` to ensure only specific words are considered. The result is a sparse matrix called `tfidf_matrix`, where each row represents a review and each column a word from our vocabulary.

To visualize this matrix, we convert it into a DataFrame using `pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)`. Now, we can see the actual TF-IDF scores for each word in each review. For example, in the first review, words like “cheap,” “design,” and “directly” have TF-IDF scores of `0.138347`, `0.098191`, and `0.220424` respectively.

This step helps us understand which words are important for each review. Once we have this word-to-weight mapping, we’re ready to build our TF-IDF weighted document embeddings.

In [9]:
tfidf_vectorizer = TfidfVectorizer(vocabulary=vocab) # calculating TF-IDF scores for the reviews
tfidf_matrix = tfidf_vectorizer.fit_transform(reviews)

feature_names = tfidf_vectorizer.get_feature_names_out() # getting the feature names (words) in the vocabulary

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names) # creating a DataFrame with the feature names as columns

# take a look at the tfidf word weights dictionary of the first review
print(tfidf_df.loc[0][tfidf_df.loc[0] != 0])  # this will print non-zero TF-IDF scores for the first review

bottom          0.104909
cheap           0.138347
design          0.098191
directly        0.220424
fact            0.146870
flaw            0.186937
found           0.110168
half            0.299921
high            0.105097
hopes           0.168447
imo             0.193098
initially       0.170978
layer           0.273897
layers          0.164018
major           0.194689
medium          0.090457
net             0.486788
nicely          0.114001
outrageously    0.256797
petite          0.179935
reordered       0.201166
sewn            0.157311
tight           0.099443
usual           0.110898
wanted          0.106882
work            0.089428
zip             0.157909
zipper          0.137967
Name: 0, dtype: float64


#### Generating TF-IDF Weighted Word Embeddings

Finally, we create our weighted embeddings using the function `get_tfidf_weighted_embeddings`. This function uses both the Word2Vec model and the computed TF-IDF scores to build weighted vectors for each review. Here’s what happens step-by-step:

- **Getting Feature Names**: We extract the feature names from the `tfidf_vectorizer`, which gives us the list of words that the TF-IDF model considers.

- **Review Loop**: We go through each review and split it into words. If a word exists in both the Word2Vec model and the TF-IDF vocabulary, we fetch its word vector from the model and get its weight using `tfidf_matrix[idx, word_index]`.

- **Weighting the Vectors**: We multiply the word vector by its TF-IDF weight `(word_vector * weight)`. This way, words with higher importance in a review (based on TF-IDF) have a stronger influence on the final embedding.

- **Summing the Weighted Vectors**: For each review, we sum up all the weighted word vectors using `np.sum(word_vectors, axis=0)` to get a single, weighted embedding.

After generating these embeddings, we check the shape again and print a sample of the first review’s embedding. This sample shows how different the TF-IDF weighted embedding is compared to the unweighted version because it takes the importance of words into account.

In [10]:
def get_tfidf_weighted_embeddings(reviews, model, tfidf_matrix, tfidf_vectorizer): # function to Compute TF-IDF Weighted Embeddings
    review_vectors = []
    feature_names = tfidf_vectorizer.get_feature_names_out()  # getting the feature names from the vectorizer
    
    for idx, review in enumerate(reviews):
        word_vectors = []
        for word in review.split():
            if word in model and word in feature_names:
                word_vector = model[word]
                word_index = feature_names.tolist().index(word)  # getting word index in TF-IDF
                weight = tfidf_matrix[idx, word_index]  # getting the TF-IDF weight for this word
                word_vectors.append(word_vector * weight)  # multiplying word vector by its weight
                
        if word_vectors:
            review_vector = np.sum(word_vectors, axis=0)  # then sum the weighted word vectors
        else:
            review_vector = np.zeros(model.vector_size)
        
        review_vectors.append(review_vector)
    return np.array(review_vectors)

weighted_embeddings = get_tfidf_weighted_embeddings(reviews, word2vec_model, tfidf_matrix, tfidf_vectorizer) # generate TF-IDF weighted embeddings
print("Generated TF-IDF weighted word embeddings. Shape:", weighted_embeddings.shape)

Generated TF-IDF weighted word embeddings. Shape: (19652, 300)


In [11]:
print("Sample TF-IDF Weighted Embedding for Review 1:\n", weighted_embeddings[0][:10])  # here we print first 10 sample of weighted embeddings values

Sample TF-IDF Weighted Embedding for Review 1:
 [-0.10499685  0.14749125 -0.20255388  0.46531036 -0.26771274 -0.40628463
  0.56281286 -0.53371066  0.51953703  0.57075858]


### Saving outputs

#### 2.1.2 Save the Count Vectors to a File
We’re opening a file named `count_vectors.txt` to save our vectors. For each review, we write its `Clothing ID` followed by a list of `word_index:count` pairs for the words that appear in the review. This step involves identifying non-zero elements in the vector `(vector.nonzero()[1])` and creating a readable format like `10:3, 45:1` (showing word 10 appeared 3 times, word 45 appeared once). This way, we capture the important information from each review and store it in a simple format for future analysis.
- count_vectors.txt

In [12]:
with open('count_vectors.txt', 'w') as out_file: # writing the count vectors to count_vectors.txt
    for i, vector in enumerate(count_features):
        review_id = data_df.index[i]
        non_zero = vector.nonzero()[1]  # getting indices of non-zero word counts here
        word_counts = [f"{word_index}:{vector[0, word_index]}" for word_index in non_zero]
        
        out_file.write(f"#{review_id}, " + ', '.join(word_counts) + '\n')  # writing the Clothing ID and the count vector in the required format

# Task 3. Clothing Review Classification

# Q1: Language model comparisons

#### Which Language Model Performed Best?
Based on the results from our 5-fold cross-validation, the **Count Vectorizer** feature representation performs the best with Logistic Regression, achieving the **highest mean accuracy of 86.48%**. This indicates that using the count-based feature representation is more effective for this classification task compared to the unweighted or weighted word embedding representations.

#### Explanation:
We used a **5-fold cross-validation** approach to evaluate each feature representation. In the code, we set up the cross-validation using the `KFold` function with `n_splits=5` to split the data into 5 equal parts (`folds`). This ensures that each part is used for testing exactly once. We then created a function called `evaluate()` that initializes a **Logistic Regression** model (`LogisticRegression(random_state=seed, max_iter=1000)`), trains it on the training data (`model.fit(X_train, y_train)`), and calculates its accuracy on the test data (`model.score(X_test, y_test)`). This function was called for each of the three feature types: `count_features`, `unweighted_embeddings`, and `weighted_embeddings`.

#### Analysis of the Results:
After running the Logistic Regression on all 5 folds, we calculated the mean accuracy for each feature type:

- **Count Vectorizer**: **86.48%**
- **Unweighted Embeddings**: **85.87%**
- **Weighted Embeddings**: **85.77%**

This means that the **Count Vectorizer** captured enough information to predict recommendations accurately without requiring the added complexity of embeddings. One reason for this is that word counts provide a straightforward representation for shorter reviews, where simple frequency-based features are enough to capture sentiment patterns.

In [13]:
data_df = pd.read_csv('processed.csv') # loading the processed CSV file containing cleaned reviews
data_df = data_df.dropna(subset=['Final Cleaned Review'])

seed = 15 # setting the seed value and initialize the 5-Fold Cross-Validation
num_folds = 5  # updating number of folds to 5
kf = KFold(n_splits=num_folds, random_state=seed, shuffle=True)  # initializing the 5-fold cross-validation
print(kf)

def evaluate(X_train, X_test, y_train, y_test, seed): # evaluating a Logistic Regression model and return its accuracy
    model = LogisticRegression(random_state=seed, max_iter=1000)  # initializing a logistic regression model here
    model.fit(X_train, y_train) # fit
    return model.score(X_test, y_test)  

cv_df = pd.DataFrame(columns=['count', 'unweighted', 'weighted'], index=range(num_folds))  # adjusted columns

recommendations = data_df['Recommended IND'].values  # target labels

fold = 0

for train_index, test_index in kf.split(list(range(len(recommendations)))): # for iteration 
   
    y_train = recommendations[train_index]  # splitting the labels based on train and test indices
    y_test = recommendations[test_index]

    # Count Features
    X_train_count, X_test_count = count_features[train_index], count_features[test_index] 
    cv_df.loc[fold, 'count'] = evaluate(X_train_count, X_test_count, y_train, y_test, seed)

    # Unweighted Embeddings
    X_train_unweighted, X_test_unweighted = unweighted_embeddings[train_index], unweighted_embeddings[test_index]
    cv_df.loc[fold, 'unweighted'] = evaluate(X_train_unweighted, X_test_unweighted, y_train, y_test, seed)

    # Weighted Embeddings
    X_train_weighted, X_test_weighted = weighted_embeddings[train_index], weighted_embeddings[test_index]
    cv_df.loc[fold, 'weighted'] = evaluate(X_train_weighted, X_test_weighted, y_train, y_test, seed)

    fold = fold + 1

print(cv_df) # here the cross validation result

# mean accuracy for each feature type
mean_accuracies = cv_df.mean()
print("\nMean Accuracies for Each Feature:")
print(mean_accuracies)

KFold(n_splits=5, random_state=15, shuffle=True)
      count unweighted  weighted
0  0.876876   0.852455  0.846858
1  0.869499   0.862376  0.865174
2  0.875827   0.858015  0.862341
3  0.871247   0.859033  0.854453
4  0.875827   0.861578  0.859796

Mean Accuracies for Each Feature:
count         0.873855
unweighted    0.858691
weighted      0.857725
dtype: object


# Q2: Does more information provide higher accuracy?

**Yes**, more information does provide higher accuracy. When we combined both the **title** and the **detailed review text**, the model achieved a mean accuracy of **88.91%**, which was higher than using only the title **(86.48%)** or only the detailed review text **(87.39%)**. This shows that combining both pieces of information provides a more complete picture of each review, allowing the model to make better predictions. The results clearly indicate that incorporating more information helps improve model performance.
- **Only the title : Accuracy (86.48)**
- **Only the detailed review text : Accuracy (87.39)**
- **Both title and detailed review : Accuracy (88.91)**

To answer this question, we compared different feature representations of clothing item reviews and their impact on classification accuracy using the Logistic Regression model. We created three distinct models using:

- 1. **Only the title** of the review
- 2. **Only the detailed review text**, which we had already done in Task 3.
- 3. **Both title and detailed review** combined to see if adding more information improves accuracy.

Let’s go through each of these models and their results to understand the impact of adding more information.

## 2.1 Only title of the review

For the first model, we focused only on the `Title` of each review. We cleaned, preprocessed, and created features using Count Vectorizer to train the model. The results of this model give us a baseline for how well just the title performs on its own.

#### 2.1.1 Preprocessing the Title Column
In this block, we performed the text cleaning operations on the `Title` column:

**What We Did**: We followed several preprocessing steps like we did in task 1 of this assignment: tokenization, lowercasing, stopword removal, removing rare words, and removing the top 20 most frequent words to create the cleaned `Final Cleaned Title column`.

In [14]:
# 1: extracting the Title column and tokenize it
titles = data_df['Title'].fillna("")  # Replace NaNs with empty strings

# 2: tokenizing it
tokenizer = RegexpTokenizer(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?") # defining the tokenizer with the specified regular expression
tokenized_titles = titles.apply(lambda x: tokenizer.tokenize(str(x)))

# 3: converting all words to lowercase
lowercased_titles = tokenized_titles.apply(lambda tokens: [token.lower() for token in tokens])

# 4: removing words with length less than 2 characters
filtered_titles = lowercased_titles.apply(lambda tokens: [token for token in tokens if len(token) > 1])

# 5: removing stopwords
with open('stopwords_en.txt', 'r') as file: # loading stopwords list from file
    stopwords_list = [line.strip() for line in file]
stopword_removed_titles = filtered_titles.apply(lambda tokens: [token for token in tokens if token not in stopwords_list])

# 6: removing words that appear only once in the entire collection
all_words = [word for review in stopword_removed_titles for word in review]
term_freq = FreqDist(all_words)
once_removed_titles = stopword_removed_titles.apply(lambda tokens: [word for word in tokens if term_freq[word] > 1])

# 7: removing the top 20 most frequent words based on document frequency
unique_words_per_review = [set(review) for review in once_removed_titles]
unique_words = [word for review in unique_words_per_review for word in review]
doc_freq = FreqDist(unique_words)
top_20_words = {word for word, freq in doc_freq.most_common(20)}

final_cleaned_titles = once_removed_titles.apply(lambda tokens: [word for word in tokens if word not in top_20_words])

# storing the final cleaned version in the specified column
data_df['Final Cleaned Title'] = final_cleaned_titles.apply(lambda x: ' '.join(x))

# displaying the first few rows of the cleaned Title column
data_df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Final Cleaned Review,Final Cleaned Title
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,high hopes wanted work initially petite usual ...,major design flaws
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,jumpsuit fun flirty fabulous time compliments,favorite buy
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,shirt due adjustable front tie length leggings...,shirt
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses,tracy reese dresses petite feet tall brand pre...,petite
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits,basket hte person store pick teh pale hte gorg...,shimmer fun


#### 2.1.2 Creating Count Vector Features for Title
In this block, we generated Count Vectorizer features using the cleaned title text:

**What We Did:** We extracted the final cleaned titles, created a vocabulary, and used `CountVectorizer` to convert the titles into numerical feature vectors. This gave us a matrix where each row corresponds to a review and each column to a unique word in the title.

In [15]:
titles = data_df['Final Cleaned Title'].tolist() # extracting the 'Final Cleaned Title' column and create a list of titles

all_words = [word for title in data_df['Final Cleaned Title'].str.split() for word in title] # generating a vocabulary from the 'Final Cleaned Title' column
vocab = {word: index for index, word in enumerate(sorted(set(all_words)))}

cVectorizer = CountVectorizer(analyzer="word", vocabulary=vocab) # initializing the CountVectorizer with the generated vocabulary

count_features = cVectorizer.fit_transform(titles)  # tansform the cleaned titles into count vector features

print(count_features.shape)

(19652, 1608)


#### 2.1.3 Training and Evaluation Using Logistic Regression
Finally, we trained a Logistic Regression model using the Count Vectorizer features and evaluated it using 5-fold cross-validation:

**Results**: The mean accuracy using only titles was **86.48%**, which gives us a solid performance using just the title information.

In [16]:
seed = 15  # setting the seed value
num_folds = 5  # number of folds for cross-validation
kf = KFold(n_splits=num_folds, random_state=seed, shuffle=True)  # initializing 5-fold cross-validation

cv_df = pd.DataFrame(columns=['count'], index=range(num_folds))  # # initializing DataFrame to store results

fold = 0
for train_index, test_index in kf.split(list(range(len(recommendations)))):  # iterating through the folds
    y_train = recommendations[train_index]  # target labels for training
    y_test = recommendations[test_index]  # target labels for testing

    X_train_count, X_test_count = count_features[train_index], count_features[test_index]  # split count features

    # training
    model = LogisticRegression(random_state=seed, max_iter=1000)
    model.fit(X_train_count, y_train)
    cv_df.loc[fold, 'count'] = model.score(X_test_count, y_test)  # store accuracy

    fold = fold + 1

print(cv_df)  # here the cross validation result

mean_accuracy = cv_df.mean() # mean accuracy
print("\nMean Accuracy:")
print(mean_accuracy)

      count
0  0.858051
1  0.860341
2  0.866412
3  0.869466
4   0.86972

Mean Accuracy:
count    0.864798
dtype: object


## 2.2 Only Description/Text of the Review
For the second experiment, we used only the detailed review text for training and evaluation. We’ve already done this in **Task 3**, where the mean accuracy achieved was **87.39%** using only the `Final Cleaned Review`. This tells us that the review text alone performs better than using only the title.

## 2.3 Combining Title and Detailed Review

Finally, we combined both the Final Cleaned Title and Final Cleaned Review into a single feature to see if more information helps boost accuracy.

#### 2.3.1 Creating Combined Features
We created a new column Combined Feature by joining the title and review text:

**What We Did:** We concatenated the cleaned title and review text for each review and used Count Vectorizer to create combined features.

In [17]:
# combining 'Final Cleaned Title' and 'Final Cleaned Review' into a single feature
data_df['Combined Feature'] = data_df['Final Cleaned Title'] + " " + data_df['Final Cleaned Review']

combined_titles_reviews = data_df['Combined Feature'].tolist() # generating Count Vectorizer Features for the combined text
cVectorizer = CountVectorizer(analyzer="word")
combined_count_features = cVectorizer.fit_transform(combined_titles_reviews)
print(combined_count_features.shape)

(19652, 7273)


#### 2.3.2 Training and Evaluation Using Combined Features
We used 5-fold cross-validation again, similar to the earlier sections, but now with the combined features:

**Results:** The mean accuracy using the combined features was **88.91%**, which shows that combining the title and review text provided a significant boost compared to using either feature alone.

In [18]:
# using Logistic Regression with 5-Fold Cross-Validation for the Combined Features
seed = 15
num_folds = 5
kf = KFold(n_splits=num_folds, random_state=seed, shuffle=True)
recommendations = data_df['Recommended IND'].values

cv_df_combined = pd.DataFrame(columns=['combined_count'], index=range(num_folds)) # initializing a DataFrame to store results

fold = 0
for train_index, test_index in kf.split(list(range(len(recommendations)))):
    y_train = recommendations[train_index]
    y_test = recommendations[test_index]

    X_train_combined, X_test_combined = combined_count_features[train_index], combined_count_features[test_index]   # splitting the combined features

    model = LogisticRegression(random_state=seed, max_iter=1000)     # training
    model.fit(X_train_combined, y_train)
    cv_df_combined.loc[fold, 'combined_count'] = model.score(X_test_combined, y_test)

    fold += 1

print(cv_df_combined)
print("\nMean Accuracy for Combined Features:", cv_df_combined['combined_count'].mean())

  combined_count
0       0.892139
1       0.889341
2       0.891603
3       0.887532
4       0.885242

Mean Accuracy for Combined Features: 0.889171425926753


**Final Conclusion:**

Adding more information (both title and detailed review) significantly improved model performance. This shows that having a more complete representation of the review helps the model capture important details, leading to better predictions.

## Summary
In this assignment, I focused on exploring how combining different parts of the review data affects model performance. I first worked on the Title feature alone, cleaning it up and converting it into numerical features. This model gave me a baseline accuracy of 86.48%. Next, I used the detailed review text, which resulted in a slightly better accuracy of 87.39%. Finally, I combined both the title and review text into a single feature, and this improved the accuracy to 88.91%.

The results clearly show that adding more context and details from both the title and the review text helped my model make better predictions. This process taught me that using more information provides the model with a fuller picture, leading to better accuracy. I now have a solid understanding of how to experiment with different text features to see their impact on a model’s performance.

## References

[1] “gensim: topic modelling for humans,” radimrehurek.com. https://radimrehurek.com/gensim/models/word2vec.html

[2] “Python String join() Method,” GeeksforGeeks, Jan. 02, 2018. https://www.geeksforgeeks.org/python-string-join-method/

[3] leo, “python combine split and join into 1 line of code,” Stack Overflow, 2024. https://stackoverflow.com/questions/53502754/python-combine-split-and-join-into-1-line-of-code (accessed Sep. 26, 2024).

[4] “toarray,” PyPI, Feb. 06, 2023. https://pypi.org/project/toarray/ (accessed Sep. 30, 2024).